In [1]:
CSV_PATH='/kaggle/input/datasets/solarmainframe/ids-intrusion-csv/02-20-2018.csv'

In [2]:
import pandas as pd
import numpy as np
import os
import gc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from IPython.display import display

2026-04-02 00:10:12.315388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775088612.619222      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775088612.730544      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775088613.470479      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775088613.470522      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775088613.470525      55 computation_placer.cc:177] computation placer alr

In [3]:

# CONFIG

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

SAMPLE_SIZE = None   
BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 3e-4

D_TOKEN = 96
N_BLOCKS = 2
ATTENTION_HEADS = 4
DROPOUT = 0.2

benign_label = "Benign"
BENIGN_RATIO = 2   

In [4]:

# LOAD AND CLEAN 

def load_and_clean_data(csv_path, sample_size=None):
    print(f"Loading data from {csv_path}...")

    df = pd.read_csv(csv_path, low_memory=False)
    display(df.describe())
    if 'Dst Port' in df.columns:
        df = df[df['Dst Port'] != 'Dst Port']

    df.columns = df.columns.str.strip()

    if 'Label' not in df.columns:
        raise ValueError("No 'Label' column found in dataset.")

    df['Label'] = df['Label'].astype(str).str.strip()

    cols_to_numeric = df.columns.drop('Label')
    df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df.dropna(axis=1, how='all', inplace=True)

    print("Fill NaNs with means...")
    for col in df.columns.drop('Label'):
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mean())

    # Drop any remaining NaNs
    df.dropna(inplace=True)

    if sample_size is not None and sample_size < len(df):
        print(f"Sampling {sample_size:,} rows...")
        df = df.sample(n=sample_size, random_state=RANDOM_STATE)

    print(f"Final shape: {df.shape}")
    print("\nClass distribution:")
    print(df['Label'].value_counts())

    return df

In [5]:

# UNDERSAMPLE TRAIN 
def undersample_benign_train_only(X_train, y_train, benign_ratio=2, random_state=42):
    """
    Keep all malicious rows.
    Keep benign rows only if benign > benign_ratio * malicious.
    """
    train_df = X_train.copy()
    train_df['BinaryLabel'] = y_train.values

    benign_df = train_df[train_df['BinaryLabel'] == 0]
    malicious_df = train_df[train_df['BinaryLabel'] == 1]

    n_benign = len(benign_df)
    n_malicious = len(malicious_df)

    
    print("TRAIN SET BEFORE UNDERSAMPLING")
    
    print(f"Benign    : {n_benign:,}")
    print(f"Malicious : {n_malicious:,}")

    max_benign_allowed = benign_ratio * n_malicious

    if n_benign > max_benign_allowed:
        print(f"\nUndersampling benign to {max_benign_allowed:,} rows...")
        benign_df = benign_df.sample(n=max_benign_allowed, random_state=random_state)
    else:
        print("\nNo undersampling needed.")

    train_balanced = pd.concat([benign_df, malicious_df], axis=0)
    train_balanced = train_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)

    X_train_bal = train_balanced.drop(columns=['BinaryLabel'])
    y_train_bal = train_balanced['BinaryLabel']

    
    print("TRAIN SET AFTER UNDERSAMPLING")
    
    print(y_train_bal.value_counts())

    return X_train_bal, y_train_bal

In [6]:

# FEATURE TOKENIZER

class FeatureTokenizer(layers.Layer):
    def __init__(self, num_features, d_token, **kwargs):
        super(FeatureTokenizer, self).__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.feature_weights = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.feature_biases = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True,
        )
        self.cls_token = self.add_weight(
            shape=(1, 1, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
        )
        super(FeatureTokenizer, self).build(input_shape)

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        x_expanded = tf.expand_dims(inputs, -1)
        tokens = x_expanded * self.feature_weights + self.feature_biases
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        return tf.concat([cls_tokens, tokens], axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_features": self.num_features,
            "d_token": self.d_token
        })
        return config

# TRANSFORMER BLOCK

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout_rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ffn = keras.Sequential([
            layers.Dense(embed_dim * 2, activation="gelu"),
            layers.Dropout(dropout_rate),
            layers.Dense(embed_dim),
        ])

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dropout_rate": self.dropout_rate
        })
        return config


# BUILD MODEL

def build_ft_transformer(num_features):
    inputs = layers.Input(shape=(num_features,))

    x = FeatureTokenizer(num_features, D_TOKEN)(inputs)

    for _ in range(N_BLOCKS):
        x = TransformerBlock(D_TOKEN, ATTENTION_HEADS, DROPOUT)(x)

    cls_output = x[:, 0, :]
    x = layers.LayerNormalization()(cls_output)
    x = layers.Dense(128, activation="gelu")(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(64, activation="gelu")(x)
    x = layers.Dropout(DROPOUT)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [7]:

# LOAD DATA

df = load_and_clean_data(CSV_PATH, sample_size=SAMPLE_SIZE)

print("\nMapping labels to binary...")
df['BinaryLabel'] = df['Label'].apply(lambda x: 0 if x == benign_label else 1)

X = df.drop(columns=['Label', 'BinaryLabel'])
y = df['BinaryLabel']

print("\nBinary class distribution:")
print(y.value_counts())

Loading data from /kaggle/input/datasets/solarmainframe/ids-intrusion-csv/02-20-2018.csv...


,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,...,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06,7.948748e+06
mean,4.527693e+04,7.364011e+03,9.363284e+00,1.353327e+07,1.698159e+01,6.713097e+00,7.137929e+02,5.008268e+03,1.706698e+02,1.329747e+01,...,1.322718e+01,1.597078e+01,2.591440e+05,1.328942e+05,3.871102e+05,1.694898e+05,5.174784e+06,1.802277e+05,5.322670e+06,5.006831e+06
std,2.112095e+04,1.727306e+04,5.247283e+00,3.243525e+07,1.244368e+03,1.580529e+02,3.983634e+04,2.281501e+05,2.599629e+02,2.421130e+01,...,1.243023e+03,6.187313e+00,3.244362e+06,1.978675e+06,4.296665e+06,2.728013e+06,1.533011e+07,1.805569e+06,1.562210e+07,1.517835e+07
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.970600e+04,5.300000e+01,6.000000e+00,4.480000e+02,1.000000e+00,1.000000e+00,1.100000e+01,0.000000e+00,1.000000e+00,0.000000e+00,...,0.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.241300e+04,8.000000e+01,6.000000e+00,5.207900e+04,2.000000e+00,1.000000e+00,4.300000e+01,1.080000e+02,4.100000e+01,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.779900e+04,3.389000e+03,1.700000e+01,3.170154e+06,7.000000e+00,5.000000e+00,3.910000e+02,9.640000e+02,1.940000e+02,3.400000e+01,...,3.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,6.553500e+04,1.700000e+01,1.200000e+08,2.800430e+05,2.580600e+04,8.961376e+06,3.758338e+07,5.647000e+03,1.460000e+03,...,2.800420e+05,4.800000e+01,1.132691e+08,7.523241e+07,1.132691e+08,1.132691e+08,1.200000e+08,7.639395e+07,1.200000e+08,1.200000e+08


Fill NaNs with means...
Final shape: (7948748, 80)

Class distribution:
Label
Benign                    7372557
DDoS attacks-LOIC-HTTP     576191
Name: count, dtype: int64

Mapping labels to binary...

Binary class distribution:
BinaryLabel
0    7372557
1     576191
Name: count, dtype: int64


In [8]:

# TRAIN / VAL / TEST SPLIT
print("\nSplitting into Train / Val / Test...")


X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nShapes BEFORE undersampling:")
print("X_train_full:", X_train_full.shape)
print("X_val       :", X_val.shape)
print("X_test      :", X_test.shape)

print("\nTrain class distribution:")
print(y_train_full.value_counts())

print("\nValidation class distribution:")
print(y_val.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())


Splitting into Train / Val / Test...

Shapes BEFORE undersampling:
X_train_full: (6358998, 79)
X_val       : (794875, 79)
X_test      : (794875, 79)

Train class distribution:
BinaryLabel
0    5898045
1     460953
Name: count, dtype: int64

Validation class distribution:
BinaryLabel
0    737256
1     57619
Name: count, dtype: int64

Test class distribution:
BinaryLabel
0    737256
1     57619
Name: count, dtype: int64


In [9]:
# UNDERSAMPLE ONLY TRAIN
X_train, y_train = undersample_benign_train_only(
    X_train_full,
    y_train_full,
    benign_ratio=BENIGN_RATIO,
    random_state=RANDOM_STATE
)

#optional
del X_train_full, y_train_full, df
gc.collect()

TRAIN SET BEFORE UNDERSAMPLING
Benign    : 5,898,045
Malicious : 460,953

Undersampling benign to 921,906 rows...
TRAIN SET AFTER UNDERSAMPLING
BinaryLabel
0    921906
1    460953
Name: count, dtype: int64


0

In [10]:
# SCALE DATA

print("\nScaling data with RobustScaler...")

feature_names = X_train.columns.tolist()

X_train_np = X_train.values.astype(np.float32)
X_val_np   = X_val[feature_names].values.astype(np.float32)
X_test_np  = X_test[feature_names].values.astype(np.float32)

y_train_np = y_train.values.astype(np.int32)
y_val_np   = y_val.values.astype(np.int32)
y_test_np  = y_test.values.astype(np.int32)

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train_np).astype(np.float32)
X_val_scaled   = scaler.transform(X_val_np).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_np).astype(np.float32)

print("Scaled shapes:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled  :", X_val_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)


Scaling data with RobustScaler...
Scaled shapes:
X_train_scaled: (1382859, 79)
X_val_scaled  : (794875, 79)
X_test_scaled : (794875, 79)


In [11]:
# BUILD MODEL

model = build_ft_transformer(num_features=X_train_scaled.shape[1])
model.summary()

I0000 00:00:1775088900.736443      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 79)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_tokenizer               │ (None, 80, 96)         │        15,264 │
│ (FeatureTokenizer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 80, 96)         │        74,784 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 80, 96)         │        74,784 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_4           │ (None, 96)             │           192 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185,761 (725.63 KB)

 Trainable params: 185,761 (725.63 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# CLASS WEIGHTS

n_benign = np.sum(y_train_np == 0)
n_malicious = np.sum(y_train_np == 1)

class_weight = {
    0: 1.0,
    1: n_benign / n_malicious
}

print("\nClass weights:")
print(class_weight)


Class weights:
{0: 1.0, 1: np.float64(2.0)}


In [13]:
# TRAIN MODEL

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

print("\nTraining FT-Transformer...")

history = model.fit(
    X_train_scaled,
    y_train_np,
    validation_data=(X_val_scaled, y_val_np),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


Training FT-Transformer...
Epoch 1/20


I0000 00:00:1775088910.548481     142 service.cc:152] XLA service 0x7eb16c003570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775088910.548519     142 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775088911.821404     142 cuda_dnn.cc:529] Loaded cuDNN version 91002


   9/5402 ━━━━━━━━━━━━━━━━━━━━ 1:21 15ms/step - accuracy: 0.5362 - loss: 0.9129  

I0000 00:00:1775088919.455501     142 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5402/5402 ━━━━━━━━━━━━━━━━━━━━ 120s 19ms/step - accuracy: 0.9864 - loss: 0.0411 - val_accuracy: 0.9991 - val_loss: 0.0019 - learning_rate: 3.0000e-04
Epoch 2/20
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 92s 17ms/step - accuracy: 0.9993 - loss: 0.0025 - val_accuracy: 0.9991 - val_loss: 0.0018 - learning_rate: 3.0000e-04
Epoch 3/20
5401/5402 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9957 - loss: 0.0206
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 92s 17ms/step - accuracy: 0.9957 - loss: 0.0206 - val_accuracy: 0.9985 - val_loss: 0.0064 - learning_rate: 3.0000e-04
Epoch 4/20
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 92s 17ms/step - accuracy: 0.9993 - loss: 0.0030 - val_accuracy: 0.9987 - val_loss: 0.0070 - learning_rate: 1.5000e-04
Epoch 5/20
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 93s 17ms/step - accuracy: 0.9988 - loss: 0.0090 - val_accuracy: 0.9991 - val_loss: 0.0015 - learning_rate: 1.5000e-04
Epoch 6/20
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 92s 17ms/step 

In [14]:
# VALIDATION EVALUATION

print("\nEvaluating on validation set...")

val_probs = model.predict(X_val_scaled, batch_size=BATCH_SIZE).ravel()
val_preds = (val_probs >= 0.5).astype(int)

macro_f1_val = f1_score(y_val_np, val_preds, average='macro')
weighted_f1_val = f1_score(y_val_np, val_preds, average='weighted')


print("VALIDATION RESULTS")

print(f"Validation Macro F1   : {macro_f1_val:.4f}")
print(f"Validation Weighted F1: {weighted_f1_val:.4f}")

print("\nValidation Confusion Matrix:")
print(confusion_matrix(y_val_np, val_preds))


Evaluating on validation set...
3105/3105 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step
VALIDATION RESULTS
Validation Macro F1   : 0.9982
Validation Weighted F1: 0.9995

Validation Confusion Matrix:
[[736877    379]
 [    11  57608]]


In [15]:
# FINAL TEST EVALUATION

print("\nEvaluating on final test set...")

test_probs = model.predict(X_test_scaled, batch_size=BATCH_SIZE).ravel()
test_preds = (test_probs >= 0.5).astype(int)

macro_f1 = f1_score(y_test_np, test_preds, average='macro')
weighted_f1 = f1_score(y_test_np, test_preds, average='weighted')

print("\n==================================================")
print(" FINAL EVALUATION: FT-TRANSFORMER ")
print("==================================================")
print(f"Macro F1 Score   : {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test_np, test_preds))

print("\nDetailed Classification Report:")
print(classification_report(
    y_test_np,
    test_preds,
    target_names=['Benign', 'Malicious'],
    digits=4
))


Evaluating on final test set...
3105/3105 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step

 FINAL EVALUATION: FT-TRANSFORMER 
Macro F1 Score   : 0.9980
Weighted F1 Score: 0.9995

Confusion Matrix:
[[736838    418]
 [    11  57608]]

Detailed Classification Report:
              precision    recall  f1-score   support

      Benign     1.0000    0.9994    0.9997    737256
   Malicious     0.9928    0.9998    0.9963     57619

    accuracy                         0.9995    794875
   macro avg     0.9964    0.9996    0.9980    794875
weighted avg     0.9995    0.9995    0.9995    794875



In [16]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

print("Evaluating on train set to complete the table...")
train_probs = model.predict(X_train_scaled, batch_size=BATCH_SIZE).ravel()
train_preds = (train_probs >= 0.5).astype(int)

train_acc = accuracy_score(y_train_np, train_preds)
val_acc = accuracy_score(y_val_np, val_preds) 
test_acc = accuracy_score(y_test_np, test_preds) 

train_f1 = f1_score(y_train_np, train_preds, average='macro')
val_f1 = macro_f1_val   
test_f1 = macro_f1      

total_instances = 7948748
benign_count = 7372557
malicious_count = 576191

benign_pct = (benign_count / total_instances) * 100
malicious_pct = (malicious_count / total_instances) * 100

summary_data = {
    "Attack Included": ["DDoS attacks-LOIC-HTTP, DDOS-HOIC"],
    "Train Accuracy": [f"{train_acc:.4f}"],
    "Val Accuracy": [f"{val_acc:.4f}"],
    "Test Accuracy": [f"{test_acc:.4f}"],
    "Train F1 (Macro)": [f"{train_f1:.4f}"],
    "Val F1 (Macro)": [f"{val_f1:.4f}"],
    "Test F1 (Macro)": [f"{test_f1:.4f}"],
    "Total Instances": [f"{total_instances:,}"],
    "Split (Train/Val/Test)": ["80% / 10% / 10%"],
    "Benign %": [f"{benign_pct:.2f}%"],
    "Malicious %": [f"{malicious_pct:.2f}%"]
}

summary_df = pd.DataFrame(summary_data)

display(summary_df)

Evaluating on train set to complete the table...
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step


,Attack Included,Train Accuracy,Val Accuracy,Test Accuracy,Train F1 (Macro),Val F1 (Macro),Test F1 (Macro),Total Instances,Split (Train/Val/Test),Benign %,Malicious %
0,"DDoS attacks-LOIC-HTTP, DDOS-HOIC",0.9996,0.9995,0.9995,0.9995,0.9982,0.9980,"7,948,748",80% / 10% / 10%,92.75%,7.25%
